# Module `algorithms/tabu_search.py`

La **recherche tabou** (*Tabu Search*, TS) est une metaheuristique introduite par Glover [1] et formalisee dans Glover & Laguna [2]. Sa structure repose sur trois principes qui la distinguent radicalement des autres metaheuristiques du projet.

**1. Best-improvement global a chaque iteration.** Contrairement a SA (qui tire **un** voisin aleatoire) ou GRASP (qui se contente d'une descente), TS **balaye tout le voisinage** a chaque iteration et choisit le meilleur mouvement. C'est plus couteux par iteration mais plus precis : on ne rate jamais un bon mouvement par malchance.

**2. Acceptation de degradations sans randomisation.** Pour sortir des optima locaux, TS accepte le meilleur mouvement **meme s'il degrade le cout**. Mais la, contrairement a SA, **pas de randomisation** : la decision est purement deterministe (le meilleur des voisins admissibles). Du coup il faut un autre mecanisme pour ne pas revenir immediatement en arriere - c'est le role de la **liste tabou**.

**3. Memoire courte (la "tabou-cite").** Apres avoir effectue un mouvement, on **interdit** pendant `tabu_tenure` iterations tout mouvement qui defait ce qu'on vient de faire. C'est ce qui empeche les **cycles** d'iterations : sans memoire, l'algo defait au pas `t+1` ce qu'il a fait au pas `t` (la degradation acceptee redevient une amelioration vue dans l'autre sens). On stocke donc des **attributs** des mouvements recents :

- Attribut tabou relocate : **client** deplace (interdit de redeplacement pendant `tabu_tenure` iterations).
- Attribut tabou swap : **paire** de clients echangee (interdite de re-echange).

Stocker des attributs plutot que des solutions completes economise enormement de memoire et permet d'interdire toute une classe de mouvements similaires (Glover & Laguna [2, chap. 3]).

**Aspiration**. Une regle stricte de tabou peut interdire un mouvement qui menerait a une nouvelle meilleure solution globale. La regle d'**aspiration par objectif** (Glover & Laguna [2, section 2.4]) leve l'interdiction si le mouvement tabou produit un cout **strictement** meilleur que le meilleur connu : `neighbor_cost < best_cost - 1e-9`.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent / "src"))

from cesipath import GraphGenerationConfig, GraphGenerator
from cesipath.algorithms import tabu_search
from cesipath.solver_input import build_static_solver_input


## 1. Appel standard


In [2]:
cfg = GraphGenerationConfig(node_count=18, seed=5)
instance = GraphGenerator(cfg).generate()
si = build_static_solver_input(instance)

sol = tabu_search(si, max_iterations=200, tabu_tenure=7, seed=0)
print(f"cout = {sol.total_cost:.2f}  ; routes = {sol.route_count}")


cout = 764.32  ; routes = 4


## 2. Parametres

| Parametre | Role |
|---|---|
| `max_iterations` | Borne dure sur le nombre d'iterations. |
| `tabu_tenure` | Duree pendant laquelle un attribut reste tabou. Plus grand = plus diversifiant, moins intensif. Voir section 3 pour le tuning. |
| `max_no_improve` | Arret precoce si aucun progres depuis N iterations - evite de gaspiller du budget sur un plateau. |
| `initial_rcl_alpha` | Construction initiale via GRASP : 0 = plus proche voisin pur, >0 = randomise. |
| `final_local_search` | Descente deterministe en post-traitement (recommande). |

**Pourquoi ces valeurs par defaut (`max_iterations=300`, `tabu_tenure=7`, `max_no_improve=80`) ?** `tabu_tenure=7` est la valeur historique recommandee par Glover [1] pour les problemes de routage et reste un excellent point de depart, comme rappele dans l'etude bibliographique de Gendreau, Hertz & Laporte [3] sur le tabou applique au VRP. Les autres valeurs sont calibrees pour rester dans des temps demo raisonnables.


## 3. Effet du `tabu_tenure`

Le tenure controle le **compromis intensification/diversification** :

- **Tenure tres faible (0 ou 1)** : aucune memoire, l'algo defait immediatement les degradations qu'il vient d'accepter et **tourne en rond** autour d'un optimum local. C'est equivalent a une descente gloutonne en plus cher.
- **Tenure trop grand** : trop de mouvements interdits a chaque iteration, l'algo ne peut plus polir une bonne piste car il se ferme lui-meme les portes des ameliorations naturelles.
- **Plage usuelle** : entre `sqrt(n)` et `n/2` ou n est le nombre de clients (regle empirique reportee par Gendreau, Hertz & Laporte [3] et Taillard [4]). Pour n=18 ici, ca donne environ 4 a 9 - ce qui colle avec le 7 recommande par Glover.

Une variante avancee (non implementee ici) est le **tenure dynamique** : on fait varier le tenure aleatoirement dans une plage `[t_min, t_max]` pour eviter les phenomenes de cycles longs (Glover & Laguna [2, section 3.2]). On ne l'a pas mis pour garder le code simple et comparable.

L'experience ci-dessous balaie plusieurs tenures - on voit bien la degradation aux extremes (`tenure=0` perd ~4%) et le plateau optimal autour de 7.


In [3]:
for tenure in (0, 3, 7, 15, 40):
    sol = tabu_search(si, max_iterations=150, tabu_tenure=tenure, max_no_improve=150, seed=1)
    print(f"tenure={tenure:>2} -> cout={sol.total_cost:.2f}  routes={sol.route_count}")


tenure= 0 -> cout=797.41  routes=4
tenure= 3 -> cout=784.29  routes=4
tenure= 7 -> cout=764.32  routes=4
tenure=15 -> cout=784.29  routes=4
tenure=40 -> cout=764.32  routes=4


## 4. Arret precoce via `max_no_improve`

**Pourquoi ce critere ?** Le balayage complet du voisinage est cher. Si l'algo n'a rien ameliore depuis longtemps, on est probablement coince dans un grand bassin sans direction d'amelioration accessible - continuer a balayer le meme voisinage est du gaspillage. `max_no_improve` est un critere d'arret **adaptatif** : on s'arrete des qu'on stagne, sans avoir besoin de connaitre a l'avance combien d'iterations seront utiles.

C'est un choix conservateur. Une alternative classique serait d'**alterner** entre intensification et **diversification longue** (forcer une serie de mouvements peu attractifs pour s'extraire du bassin), comme propose par Glover & Laguna [2, chap. 4]. On ne l'a pas implementee ici.

L'experience ci-dessous montre l'effet : un `max_no_improve` faible accelere fortement (3-7x) sans degrader la solution sur cette instance, parce que l'algo a effectivement trouve son optimum tres tot.


In [4]:
import time

for cap in (20, 80, 500):
    t0 = time.perf_counter()
    sol = tabu_search(si, max_iterations=500, tabu_tenure=7, max_no_improve=cap, seed=2)
    dt = time.perf_counter() - t0
    print(f"max_no_improve={cap:>3} -> cout={sol.total_cost:.2f}  temps={dt*1000:.1f} ms")


max_no_improve= 20 -> cout=764.32  temps=9.3 ms
max_no_improve= 80 -> cout=764.32  temps=17.8 ms
max_no_improve=500 -> cout=764.32  temps=63.3 ms


## 5. Aspiration : debloquer un mouvement tabou

**Pourquoi cette regle ?** Le statut tabou s'applique a un **attribut** (un client, une paire), pas a une solution complete. Donc deux mouvements differents peuvent partager le meme attribut tabou - et l'un d'eux peut tres bien mener a une solution **jamais visitee** et **meilleure que tout ce qu'on a vu**. Refuser ce mouvement par exces de zele serait absurde.

La regle d'**aspiration par objectif** (Glover & Laguna [2, section 2.4]) leve l'interdiction tabou si le mouvement produit un cout **strictement** meilleur que `best_cost`. Le strict (`< best_cost - 1e-9` plutot que `<=`) sert deux choses : eviter les egalites numeriques fortuites en virgule flottante, et garantir que l'aspiration n'est declenchee que pour un **vrai** progres.

Implemente dans les deux blocs de selection (relocate et swap) via la condition `aspirated = neighbor_cost < best_cost - 1e-9`. C'est la regle d'aspiration la plus simple et la plus universelle ; il en existe des plus fines (par direction de mouvement, par region de l'espace) detaillees dans Glover & Laguna [2, chap. 4].


## 6. Comparaison rapide avec GRASP et SA

**Lecture des resultats** : ces trois algos partagent la meme recherche locale finale, donc on compare vraiment leurs **strategies d'exploration** plus que la qualite de leurs polissages.

- **GRASP** : multi-depart sans memoire. Cout par iteration faible (juste construction + descente), mais besoin de plusieurs iterations pour diversifier.
- **SA** : trajectoire unique, mouvements aleatoires. Cout par iteration **tres** faible (un seul voisin tire), mais besoin de beaucoup d'iterations (>1000) pour converger via le refroidissement.
- **TS** : trajectoire unique, balayage complet. Cout par iteration **eleve** (O(n^2) mouvements evalues), mais convergence rapide en nombre d'iterations.

Le tabou est generalement le plus precis sur des instances petites/moyennes parce que son balayage exhaustif ne rate aucun mouvement local. Sur de grandes instances, le cout O(n^2) par iteration le penalise et SA ou un GA peuvent etre plus efficaces a budget temps egal - c'est ce que confirment les comparaisons systematiques de Cordeau et al. [5, section 5] sur le VRP.

Les chiffres ci-dessous (n=18) ne reflètent qu'une seule instance ; voir `benchmark.ipynb` pour des comparaisons multi-instances.


In [5]:
from cesipath.algorithms import grasp, simulated_annealing

import time

def run(name, fn, **kwargs):
    t0 = time.perf_counter()
    sol = fn(si, seed=0, **kwargs)
    dt = time.perf_counter() - t0
    print(f"{name:<6} cout={sol.total_cost:>7.2f}  temps={dt*1000:>7.1f} ms")

run("grasp",  grasp,              max_iterations=30, rcl_alpha=0.3)
run("sa",     simulated_annealing, max_iterations=2000, cooling_rate=0.97)
run("tabu",   tabu_search,         max_iterations=150, tabu_tenure=7)


grasp  cout= 764.32  temps=   19.6 ms
sa     cout= 797.41  temps=    4.3 ms
tabu   cout= 764.32  temps=   11.5 ms


## 7. Reproductibilite

Les seuls tirages aleatoires de l'algorithme sont la construction initiale (RCL si `initial_rcl_alpha > 0`) ; toute la boucle tabou est **strictement deterministe** (best-improvement sur attributs explicites). Le `random.Random(seed)` est donc en pratique surtout utile pour la phase d'initialisation. A `seed` egal, l'algorithme produit **exactement** la meme trajectoire et donc la meme solution finale.

---

## References

[1] **Glover, F. (1989).** *Tabu Search - Part I.* ORSA Journal on Computing, 1(3), 190-206. https://doi.org/10.1287/ijoc.1.3.190

[2] **Glover, F. & Laguna, M. (1997).** *Tabu Search.* Kluwer Academic Publishers. https://doi.org/10.1007/978-1-4615-6089-0

[3] **Gendreau, M., Hertz, A. & Laporte, G. (1994).** *A tabu search heuristic for the vehicle routing problem.* Management Science, 40(10), 1276-1290. https://doi.org/10.1287/mnsc.40.10.1276

[4] **Taillard, E. (1993).** *Parallel iterative search methods for vehicle routing problems.* Networks, 23(8), 661-673. https://doi.org/10.1002/net.3230230804

[5] **Cordeau, J.-F., Gendreau, M., Laporte, G., Potvin, J.-Y. & Semet, F. (2002).** *A guide to vehicle routing heuristics.* Journal of the Operational Research Society, 53(5), 512-522. https://doi.org/10.1057/palgrave.jors.2601319


In [6]:
a = tabu_search(si, max_iterations=80, tabu_tenure=7, seed=9)
b = tabu_search(si, max_iterations=80, tabu_tenure=7, seed=9)
print("identiques ?", a.total_cost == b.total_cost and a.routes == b.routes)


identiques ? True
